In [1]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg2zn11_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 39 atoms, Mg6Zn33


In [2]:
slab = surface(alloy, (2, 1, 1), 6, vacuum=8)
sc = make_supercell(slab, [[2,0,0],[0,2,0],[0,0,1]])

sym = np.array(sc.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = sc.get_positions()[:, 2]
thick = z.max() - z.min()
area = np.linalg.norm(np.cross(sc.cell[0], sc.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(2,1,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(sc)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")
print(f"Stoichiometric: {'YES' if mg*11 == zn*2 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 936, Mg: 144, Zn: 792
Thickness: 21.3 A
Area: 743.4 A²
Stoichiometric: YES
Symmetric: False


In [3]:
z = sc.get_positions()[:, 2]
sym = np.array(sc.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Tolerance: {tol:.3f} A")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

n = len(layers)
z_min, z_max = z.min(), z.max()
print(f"Total layers: {n}")

print("\nSearching for symmetric removals...\n")

found_any = False
for bot_rm in range(0, min(15, n//2)):
    for top_rm in range(0, min(15, n//2)):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) < 100:
            continue
        tr = sc[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(2,1,1), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                sym_tr = np.array(tr.get_chemical_symbols())
                mg_tr = sum(sym_tr == 'Mg')
                zn_tr = sum(sym_tr == 'Zn')
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"Mg{mg_tr} Zn{zn_tr}, removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
                found_any = True
        except:
            continue

if not found_any:
    print("No symmetric sub-slab found")

Tolerance: 0.039 A
Total layers: 163

Searching for symmetric removals...

  bot_rm=1, top_rm=1: 924 atoms, Mg144 Zn780, removed 8+4=12
  bot_rm=2, top_rm=2: 916 atoms, Mg144 Zn772, removed 12+8=20
  bot_rm=3, top_rm=3: 900 atoms, Mg136 Zn764, removed 20+16=36
  bot_rm=4, top_rm=4: 892 atoms, Mg136 Zn756, removed 24+20=44
  bot_rm=5, top_rm=5: 884 atoms, Mg136 Zn748, removed 28+24=52
  bot_rm=6, top_rm=6: 876 atoms, Mg136 Zn740, removed 32+28=60
  bot_rm=7, top_rm=7: 868 atoms, Mg128 Zn740, removed 36+32=68
  bot_rm=8, top_rm=8: 852 atoms, Mg128 Zn724, removed 44+40=84
  bot_rm=9, top_rm=9: 844 atoms, Mg120 Zn724, removed 48+44=92
  bot_rm=10, top_rm=10: 836 atoms, Mg120 Zn716, removed 52+48=100
  bot_rm=11, top_rm=11: 828 atoms, Mg120 Zn708, removed 56+52=108
  bot_rm=12, top_rm=12: 820 atoms, Mg120 Zn700, removed 60+56=116
  bot_rm=13, top_rm=13: 788 atoms, Mg120 Zn668, removed 76+72=148
  bot_rm=14, top_rm=14: 780 atoms, Mg120 Zn660, removed 80+76=156


In [4]:
removed_bot = layer_idx[0]
removed_top = layer_idx[n-1]
removed_all = removed_bot + removed_top

keep = []
for j in range(1, n - 1):
    keep.extend(layer_idx[j])

trimmed = sc[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

ps_orig = AseAtomsAdaptor().get_structure(sc)

# Find bot-top pairs
paired_bt = []
for j in removed_bot:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    for k in removed_top:
        if np.linalg.norm(sc.positions[k] - inv_cart) < 0.5:
            paired_bt.append((j, k))
            break

used_bot = set(b for b, _ in paired_bt)
used_top = set(t for _, t in paired_bt)
unpaired_bot = [j for j in removed_bot if j not in used_bot]
unpaired_top = [j for j in removed_top if j not in used_top]
remaining = unpaired_bot + unpaired_top

print(f"\nRemoved: {len(removed_all)} atoms (Mg{sum(sym[j]=='Mg' for j in removed_all)} Zn{sum(sym[j]=='Zn' for j in removed_all)})")
print(f"Bot: {len(removed_bot)}, Top: {len(removed_top)}")
print(f"Paired bot<->top: {len(paired_bt)}")
print(f"Unpaired: {len(remaining)}")

# Show unpaired with inversion positions
print(f"\nUnpaired atoms:")
print(f"{'idx':>5} {'el':>3} {'x':>8} {'y':>8} {'z':>8}  ->  {'inv_x':>8} {'inv_y':>8} {'inv_z':>8}")
print("-" * 75)
for j in remaining:
    pos = sc.positions[j]
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    print(f"{j:>5} {sym[j]:>3} {pos[0]:>8.3f} {pos[1]:>8.3f} {pos[2]:>8.3f}  ->  "
          f"{inv_cart[0]:>8.3f} {inv_cart[1]:>8.3f} {inv_cart[2]:>8.3f}")

# Check mutual pairs
print("\nMutual pairs:")
used = set()
mutual = []
for i, j1 in enumerate(remaining):
    if j1 in used:
        continue
    frac1 = ps_orig[j1].frac_coords
    inv1 = ps_orig.lattice.get_cartesian_coords((inv_translation - frac1) % 1.0)
    for j2 in remaining[i+1:]:
        if j2 in used:
            continue
        d = np.linalg.norm(sc.positions[j2] - inv1)
        if d < 1.0:
            mutual.append((j1, j2))
            used.add(j1)
            used.add(j2)
            print(f"  {j1} <-> {j2} (dist={d:.3f})")
            break

still_unpaired = [j for j in remaining if j not in used]
print(f"\nMutual pairs: {len(mutual)}, Still unpaired: {len(still_unpaired)}")

SG = P-1
Inversion: f -> [0. 0. 0.] - f

Removed: 12 atoms (Mg0 Zn12)
Bot: 8, Top: 4
Paired bot<->top: 4
Unpaired: 4

Unpaired atoms:
  idx  el        x        y        z  ->     inv_x    inv_y    inv_z
---------------------------------------------------------------------------
  241  Zn   -1.948   14.313    8.000  ->    25.320    4.771   29.336
  475  Zn   25.320    4.771    8.000  ->    -1.948   14.313   29.336
    7  Zn    5.843    4.771    8.000  ->    17.530   14.313   29.336
  709  Zn   17.530   14.313    8.000  ->     5.843    4.771   29.336

Mutual pairs:

Mutual pairs: 0, Still unpaired: 4


In [5]:
sc_fixed = sc.copy()

move_pairs = [
    (241, 475),   # move 475 to inv(241)
    (7, 709),     # move 709 to inv(7)
]

for keep_idx, move_idx in move_pairs:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    old = sc_fixed.positions[move_idx].copy()
    sc_fixed.positions[move_idx] = inv_cart
    print(f"Moved {move_idx} -> inv({keep_idx}): "
          f"({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
          f"({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(2,1,1), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if mg_f*11 == zn_f*2 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")

Moved 475 -> inv(241): (25.320, 4.771, 8.000) -> (25.320, 4.771, 29.336)
Moved 709 -> inv(7): (17.530, 14.313, 8.000) -> (17.530, 14.313, 29.336)

Atoms: 936, Mg: 144, Zn: 792
Stoichiometric: YES
Symmetric: True
Thickness: 21.3 A
Area: 743.4 A²


In [6]:
ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg2zn11_211_6layers_reconstructed_ase.data")

print(f"Saved: slabs/mg2zn11_211_6layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg2zn11_211_6layers_reconstructed_ase.data
  Atoms: 936
  Thickness: 21.3 A
  Area: 743.4 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [7]:
#unrelaxed surface energy calculation
E_slab = -994.69608       # eV
E_bulk_per_fu =  -44.119792 / 3  # eV per formula unit 
n = 936 / 13                 # formula units in slab = 32
A = 743.4           # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.706597 eV
n * E_bulk = -1058.87501 eV
E_slab - n*E_bulk = 64.17893 eV
S = 0.043166 eV/Ang^2
S = 0.6916 J/m^2


In [8]:
#relaxed surface energy calculation
E_slab_relaxed = -1003.28131277998  # eV
E_bulk_per_fu = -44.119792 / 3  # eV per formula unit
n = 936 / 13                      # 32 formula units
A = 743.4                # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 =  0.6916
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 55.59370 eV
S relaxed = 0.037392 eV/Ang^2
S relaxed = 0.5991 J/m^2

Unrelaxed S = 0.6916 J/m^2
Relaxed S   = 0.5991 J/m^2
Relaxation energy = 0.0925 J/m^2
